# Example 3.6.3 / 1: Inspect a Solved Model

Companion notebook for **3.6.3 Slacks and Active Constraints** in the AMPL lab Documentation, section "Inspect a Solved Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Inspect a Solved Model

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0;                        # Activity worth 3 per unit
var y >= 0;                        # Activity worth 2 per unit
maximize Value: 3 * x + 2 * y;     # Total value of both activities
subject to Capacity: x + y <= 10;  # Shared limit on x and y together
subject to Mix: x <= 6;            # Separate limit on x alone

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
# This model has no external data.

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
for name in ["Capacity", "Mix"]:
    c = ampl.con[name]
    print(name, c.body(), c.ub(), c.slack())

Capacity 10.0 10.0 0.0
Mix 6.0 6.0 0.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
# This model has no external data.

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Capacity.body, Capacity.ub, Capacity.slack;
display Mix.body, Mix.ub, Mix.slack;

HiGHS 1.14.0: optimal solution; objective 26
0 simplex iterations
0 barrier iterations
Capacity.body = 10
Capacity.ub = 10
Capacity.slack = 0

Mix.body = 6
Mix.ub = 6
Mix.slack = 0

